# Coleta de Dados de Hand Landmarks (Tasks API)

Este notebook utiliza a API moderna `mp.tasks` do MediaPipe para extrair coordenadas (x, y, z) dos hand landmarks e salvá-las em um CSV.

### Instruções:
1. **Defina o nome:** Altere a variável `GESTURE_NAME` na célula correspondente abaixo para o nome do gesto que deseja coletar (ex: 'vitoria', 'legal', 'pare').
2. **Inicie a webcam:** Execute a célula do loop.
3. **Grave os dados:**
   - Pressione **'r'** para alternar (toggle) a gravação automática. Quando ativado, cada frame onde sua mão aparecer será salvo no CSV.
   - Alternativamente, pressione de **0 a 9** para capturar apenas um único frame com essa label.
   - No topo da tela do vídeo, você verá o status **[REC]** ou **[STANDBY]**.
   - Pressione **'q'** para encerrar e fechar a webcam.

In [96]:
import cv2
import mediapipe as mp
import csv
import os
import numpy as np
import time

## Configuração do Reconhecedor e do CSV

In [97]:
model_path = 'models/gesture_recognizer.task'
csv_file = 'hand_landmarks_data.csv'

# Configuração do MediaPipe Tasks
BaseOptions = mp.tasks.BaseOptions
GestureRecognizer = mp.tasks.vision.GestureRecognizer
GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = GestureRecognizerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

recognizer = GestureRecognizer.create_from_options(options)

# Cabeçalho: label, x0, y0, z0, ..., x20, y20, z20
header = ['label']
for i in range(21):
    header.extend([f'x{i}', f'y{i}', f'z{i}'])

if not os.path.exists(csv_file):
    with open(csv_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(header)
    print(f"Arquivo {csv_file} criado.")
else:
    print(f"Arquivo {csv_file} já existe. Dados serão anexados.")

Arquivo hand_landmarks_data.csv já existe. Dados serão anexados.


## Funções de Desenho

In [98]:
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Polegar
    (0, 5), (5, 6), (6, 7), (7, 8),      # Indicador
    (5, 9), (9, 10), (10, 11), (11, 12),  # Médio
    (9, 13), (13, 14), (14, 15), (15, 16), # Anelar
    (0, 17), (17, 18), (18, 19), (19, 20), # Mindinho
    (5, 9), (9, 13), (13, 17)             # Base da palma
]

def draw_landmarks(image, hand_landmarks):
    h, w, _ = image.shape
    points = []
    for lm in hand_landmarks:
        points.append((int(lm.x * w), int(lm.y * h)))

    for start, end in HAND_CONNECTIONS:
        cv2.line(image, points[start], points[end], (0, 255, 0), 2)
    for p in points:
        cv2.circle(image, p, 5, (255, 0, 255), -1)
    return image

def draw_text_with_outline(image, text, position, font_scale, color, thickness=2):
    """Desenha texto com um contorno preto para aumentar o contraste."""
    font = cv2.FONT_HERSHEY_SIMPLEX
    # Desenha o contorno preto (espessura maior)
    cv2.putText(image, text, position, font, font_scale, (0, 0, 0), thickness + 2, cv2.LINE_AA)
    # Desenha o texto colorido por cima
    cv2.putText(image, text, position, font, font_scale, color, thickness, cv2.LINE_AA)

## Definição do Gesto Atual

Altere o nome abaixo antes de iniciar a captura de um novo gesto.

In [99]:
GESTURE_NAME = "te amo"  # Substitua pelo nome do seu gesto (ex: 'joinha', 'pare', etc.)

## Loop de Captura e Salvamento

In [100]:
cap = cv2.VideoCapture(0)
samples_count = 0
is_recording = False

print(f"Webcam ativa. Gesto atual: {GESTURE_NAME}")
print("Comandos: 'r' para REC (Toggle), '0-9' para single shot, 'q' para sair.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    timestamp_ms = int(time.time() * 1000)

    # Reconhecimento
    result = recognizer.recognize_for_video(mp_image, timestamp_ms)

    current_landmarks_list = None
    
    if result.hand_landmarks:
        # Pega a primeira mão detectada
        hand_landmarks = result.hand_landmarks[0]
        frame = draw_landmarks(frame, hand_landmarks)
        
        # Prepara a lista de coordenadas para o CSV
        current_landmarks_list = []
        for lm in hand_landmarks:
            current_landmarks_list.extend([lm.x, lm.y, lm.z])

    # --- LÓGICA DE SALVAMENTO --- #
    key = cv2.waitKey(5) & 0xFF

    # Atalho 'r' para toggle recording
    if key == ord('r'):
        is_recording = not is_recording
        print(f"Gravação: {'ATIVADA' if is_recording else 'DESATIVADA'}")

    # Salva automaticamente se estiver gravando e houver uma mão detectada
    if is_recording and current_landmarks_list:
        with open(csv_file, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([GESTURE_NAME] + current_landmarks_list)
        samples_count += 1

    # Atalho 0-9 para captura única (opcional)
    if ord('0') <= key <= ord('9') and current_landmarks_list:
        label = chr(key)
        with open(csv_file, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([label] + current_landmarks_list)
        samples_count += 1
        print(f"Amostra única salva como {label}")

    # --- UI DE FEEDBACK NO VÍDEO --- #
    # Status de Gravação
    rec_status = "[REC]" if is_recording else "[STANDBY]"
    rec_color = (0, 0, 255) if is_recording else (0, 255, 0) # Vermelho para REC, Verde para Standby
    
    draw_text_with_outline(frame, f"{rec_status} Gesto: {GESTURE_NAME}", (10, 40), 0.8, rec_color, 2)
    draw_text_with_outline(frame, f"Amostras Totais: {samples_count}", (10, 80), 0.7, (255, 255, 255), 2)

    cv2.imshow('Coleta de Dados - MediaPipe Tasks', frame)

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
recognizer.close()
print(f"Processo finalizado. {samples_count} amostras no arquivo {csv_file}.")

Webcam ativa. Gesto atual: te amo
Comandos: 'r' para REC (Toggle), '0-9' para single shot, 'q' para sair.
Gravação: ATIVADA
Gravação: DESATIVADA
Gravação: ATIVADA
Gravação: DESATIVADA
Processo finalizado. 155 amostras no arquivo hand_landmarks_data.csv.
